# Appendix: Validation and R Handoff

Additional validation checks and R-integration code preserved from the original workflow.


#Appendix

##R Code and Andrew's Testing

In [ ]:
import os
import time
import math
import pandas as pd
import isodate
from tqdm import tqdm
from typing import List, Dict
from googleapiclient.discovery import build

import time
import json
import kafka
from kafka import KafkaProducer
from kafka.errors import KafkaError

import io
import avro.schema
from avro.io import DatumWriter

In [ ]:
# Download the two necessary files from YouNiverse dataset
!wget "https://zenodo.org/records/4650046/files/df_channels_en.tsv.gz?download=1" -O df_channels_en.tsv.gz

# This will take a while
!wget "https://zenodo.org/records/4650046/files/yt_metadata_helper.feather?download=1" -O yt_metadata_helper.feather

In [ ]:
# Load channels
channels_df = pd.read_csv("df_channels_en.tsv.gz", sep="\t", compression="gzip")
print(channels_df.columns.tolist())
# ['channel', 'subscribers_cc', 'videos_cc', 'subscriber_rank_sb', 'weights', 'category', ...]

# Load video metadata helper (fast feather format)
videos_df = pd.read_feather("yt_metadata_helper.feather")
print(videos_df.columns.tolist())
# ['display_id', 'channel_id', 'upload_date', 'duration', 'view_count', 'like_count', ...]

##Verification

In [ ]:
# Channels dataframe
print("=== CHANNELS ===")
print(f"Shape: {channels_df.shape}")
print(f"\nColumns:\n{channels_df.columns.tolist()}")
print(f"\nDtypes:\n{channels_df.dtypes}")
print(f"\nFirst 3 rows:\n{channels_df.head(3)}")

# Videos dataframe
print("\n=== VIDEOS ===")
print(f"Shape: {videos_df.shape}")
print(f"\nColumns:\n{videos_df.columns.tolist()}")
print(f"\nDtypes:\n{videos_df.dtypes}")
print(f"\nFirst 3 rows:\n{videos_df.head(3)}")

In [ ]:
# How complete is each column?
def field_report(df, name):
    total = len(df)
    report = pd.DataFrame({
        "dtype":    df.dtypes,
        "non_null": df.notna().sum(),
        "null_pct": (df.isna().sum() / total * 100).round(2),
        "n_unique": df.nunique(),
        "sample":   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
    })
    print(f"\n=== {name} field report ===")
    print(report.to_string())

field_report(channels_df, "channels")
field_report(videos_df, "videos")

In [ ]:
# --- Date coverage ---
videos_df["upload_date"] = pd.to_datetime(videos_df["upload_date"], errors="coerce")
print("Date range:", videos_df["upload_date"].min(), "→", videos_df["upload_date"].max())
print("Unparseable dates:", videos_df["upload_date"].isna().sum())

# --- Duration ---
print("\nDuration (seconds) summary:")
print(videos_df["duration"].describe())
print("Null durations:", videos_df["duration"].isna().sum())

# --- Engagement metrics ---
for col in ["view_count", "like_count", "comment_count"]:
    if col in videos_df.columns:
        print(f"\n{col}:")
        print(videos_df[col].describe())

# --- Channel ID linkage ---
yt_channels   = set(videos_df["channel_id"].dropna().unique())
meta_channels = set(channels_df["channel"].dropna().unique())  # column may be named 'channel'
print(f"\nChannels in videos:   {len(yt_channels):,}")
print(f"Channels in metadata: {len(meta_channels):,}")
print(f"Overlap:              {len(yt_channels & meta_channels):,}")

In [ ]:
videos_window_df = videos_df[
    (videos_df["upload_date"] >= "2010-01-01") &
    (videos_df["upload_date"] <= "2011-12-31")
].copy()

print(f"Videos in 2010-2011 window: {len(videos_window_df):,}")
print(f"Unique channels in window:  {videos_window_df['channel_id'].nunique():,}")

videos_window_df["year_month"] = videos_window_df["upload_date"].dt.to_period("M")
print("\nMonthly video counts:")
print(videos_window_df.groupby("year_month").size().to_string())

In [ ]:
videos_window_df["is_long"] = (videos_window_df["duration"] > LONG_VIDEO_SECONDS).astype(int)

treated_channels = (
    videos_window_df[
        (videos_window_df["upload_date"] >= POLICY_DATE) &
        (videos_window_df["is_long"] == 1)
    ]["channel_id"]
    .unique()
)

print(f"Treated channels:   {len(treated_channels):,}")
print(f"Untreated channels: {videos_window_df['channel_id'].nunique() - len(treated_channels):,}")

long_vids = videos_window_df[videos_window_df["is_long"] == 1]["duration"] / 60

In [ ]:
panel_check = (
    videos_window_df.groupby(["channel_id", videos_window_df["upload_date"].dt.to_period("M")])
    .agg(
        video_count      = ("display_id", "count"),
        long_video_count = ("is_long",    "sum"),
        total_views      = ("view_count", "sum")
    )
    .reset_index()
)

##R Code Integration

In [ ]:
# Install R and rpy2 in Colab
!apt-get install -y r-base r-base-dev
!pip install rpy2

# Load the rpy2 extension
%load_ext rpy2.ipython

In [ ]:
%%R
# Install pacman and packages
install.packages("pacman", repos="https://cloud.r-project.org")
pacman::p_load(tidyverse, lubridate, skimr, psych, moments, broom, lmtest, zoo)

In [ ]:
%%R
# Load data (n=2000)
df <- read.csv("/content/drive/MyDrive/Classes/MIS 584/data/youtube_2000_video_sample.csv")

# Factoring
df$definition       <- factor(df$definition)
df$caption          <- factor(df$caption)
df$licensed_content <- factor(df$licensed_content)
df$category_id      <- factor(df$category_id)

# Duration parsing
df$duration_sec <- as.numeric(lubridate::duration(df$duration))

# Date features
df$published_at  <- as.POSIXct(df$published_at, format="%Y-%m-%dT%H:%M:%SZ", tz="UTC")
df$publish_year  <- factor(lubridate::year(df$published_at))
df$publish_month <- factor(lubridate::month(df$published_at))
df$publish_wday  <- factor(weekdays(df$published_at))

summary(df)

##Creating the Panel Data

In [ ]:
# Build full panel from YouNiverse window
POLICY_DATE        = pd.Timestamp("2010-12-01")
LONG_VIDEO_SECONDS = 15 * 60

window["is_long"] = (window["duration"] > LONG_VIDEO_SECONDS).astype(int)

first_treat = (
    window[(window["upload_date"] >= POLICY_DATE) & (window["is_long"] == 1)]
    .groupby("channel_id")["upload_date"]
    .min()
    .rename("first_long_video_post_policy_date")
    .reset_index()
)

panel_df = (
    window
    .groupby(["channel_id", window["upload_date"].dt.to_period("M").rename("year_month")])
    .agg(
        monthly_video_count      = ("display_id", "count"),
        monthly_long_video_count = ("is_long",    "sum"),
        monthly_views_sum_current = ("view_count", "sum"),
        monthly_likes_sum_current = ("like_count", "sum"),
        avg_duration_seconds     = ("duration",   "mean")
    )
    .reset_index()
    .merge(first_treat, on="channel_id", how="left")
)

panel_df["treated_creator"] = panel_df["first_long_video_post_policy_date"].notna().astype(int)
panel_df["month_date"]      = panel_df["year_month"].dt.to_timestamp()
panel_df["post_policy"]     = (panel_df["month_date"] >= POLICY_DATE).astype(int)
panel_df["did"]             = panel_df["treated_creator"] * panel_df["post_policy"]

print(f"Panel rows: {len(panel_df):,}")

In [ ]:
# Final export for R handoff
panel_df.to_csv("/content/drive/MyDrive/Classes/MIS 584/Project Colab Output/panel_df.csv", index=False)

print("Columns exported:", panel_df.columns.tolist())
print("Rows:", len(panel_df))